# Investigating the impact of structural and temporal behaviors in Ethereum phishing users detection* |

This project focuses on detecting phishing addresses on the Ethereum blockchain using machine learning techniques.

The study explores transactional, structural, and temporal characteristics of Ethereum addresses to identify patterns that distinguish phishing addresses from legitimate addresses.

The project covers data preparation, feature engineering, exploratory analysis, machine learning classification, model evaluation, feature analysis, temporal analysis, and graph-based learning.

The overall goal is to investigate how machine learning and graph-based approaches can be applied to Ethereum phishing detection.


## 0. Setup

In [ ]:
# install pandas, numpy, networkx, scikit-learn, xgboost, lightgbm, torch, scipy, seaborn.
!pip install -q catboost imbalanced-learn torch_geometric

In [ ]:
import os, gc, time, random, warnings, pickle, glob
from pathlib import Path
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60, "display.width", 200)
sns.set_theme(style="whitegrid")
print("pandas", pd.__version__, "| networkx", nx.__version__)

In [ ]:
PKL_PATH  = "/kaggle/input/datasets/calvinnova/muldigraph/MulDiGraph.pkl"
DATA_PATH = "/kaggle/working/transactions.csv"   # created from the pickle by the conversion cell
CACHE_DIR = "/kaggle/working/cache"

# 0.1 Data loading 
and inspection of the graph

In [ ]:

files = glob.glob("/kaggle/input/**/*", recursive=True)
print(files)

PKL_PATH = next(f for f in files if f.lower().endswith((".pkl", ".pickle")))
print("using:", PKL_PATH)

G = pd.read_pickle(PKL_PATH)
print(type(G))
print("nodes:", G.number_of_nodes(), " edges:", G.number_of_edges())
print(list(G.nodes(data=True))[:3])
print(list(G.edges(data=True))[:3])

0.2 convert the graph to csv


In [ ]:
import csv, itertools, os

DATA_PATH = "/kaggle/working/transactions.csv"
if os.path.exists(DATA_PATH):
    os.remove(DATA_PATH)                      # delete the broken 20 GB file

lab = {n: d.get("isp", 0) for n, d in G.nodes(data=True)}

n_written = 0
with open(DATA_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["from", "to", "amount", "timestamp", "FromIsPhi", "ToIsPhi"])
    edges = iter(G.edges(data=True))          # <-- the fix: a real iterator
    while True:
        chunk = list(itertools.islice(edges, 1_000_000))
        if not chunk:
            break
        w.writerows((u, v, d["amount"], int(d["timestamp"]), lab[u], lab[v])
                    for u, v, d in chunk)
        n_written += len(chunk)
        print(n_written, end="\r")

print("\nrows written:", n_written, "(expected 13,551,303)")
print("size GB:", os.path.getsize(DATA_PATH) / 1e9)
del G, lab

# 0.3 settings

In [ ]:
# ============================== USER SETTINGS ==============================
ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    CACHE_DIR = "/kaggle/working/cache"
    _csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    DATA_PATH = _csv[0] if _csv else "/kaggle/working/transactions.csv"
else:
    DATA_PATH = "/content/drive/MyDrive/ethereum_phishing/transactions.csv"
    CACHE_DIR = "/content/drive/MyDrive/ethereum_phishing/cache"
print("DATA_PATH:", DATA_PATH, "| CACHE_DIR:", CACHE_DIR)
SEED      = 42

QUICK_MODE = True   # True 

# ---- Data
WINDOW_START, WINDOW_END = "2016-11-19", "2019-01-17"
# ---- Experimental protocol
TEST_SIZE   = 0.20
CV_FOLDS    = 10
CV_REPEATS  = 1 if QUICK_MODE else 10
N_DATASETS  = 1 if QUICK_MODE else 5          # D0..D4
# ---- Feature-computation choices ----
HITS_MODE        = "authority"
TS_GRANULARITY   = "second"
ACTIVITY_FORMULA = "as_printed"
RATIO_FORMULA    = "eq7"
# ---- Classifier choices ----
RUN_SAMPLED_CV = False
SCALE_LINEAR_MODELS = False
# ---- experiment sizes ----
TRIAL_COUNT, RUNS_PER_TRIAL = (3, 5) if QUICK_MODE else (10, 10)
GENUINE_TX_SIZES = [5000, 10000, 20000] if QUICK_MODE else [5000, 10000, 20000, 30000, 40000, 50000, 60000, 80000, 90000, 100000]
# ---- GNN section ----
GNN_HIDDEN, GNN_EPOCHS, GNN_LR, GNN_WD, GNN_DROPOUT = 64, (100 if QUICK_MODE else 200), 0.01, 5e-4, 0.5
GNN_LAYERS   = [2, 3, 4, 5, 6, 7, 8]
GNN_SEEDS    = 1 if QUICK_MODE else 3
GNN_SYMMETRISE = True
GNN_LOG1P      = False
# ===========================================================================

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
def set_seed(s):
    random.seed(s); np.random.seed(s)
set_seed(SEED)
DAY = 86_400

## 1. Transaction Data Processing and Dataset Preparation



1. every phishing-related transaction, i.e. categories 0–1, 1–0 and 1–1 (≈ 60 k rows), and
2. only the row ids and days of the 0–0 transactions (used to draw the balanced samples).

A second pass then extracts just the sampled 0–0 rows.

In [ ]:
COLS = ["from", "to", "amount", "timestamp", "FromIsPhi", "ToIsPhi"]
_ALIAS = {"from": "from", "to": "to", "amount": "amount", "timestamp": "timestamp",
          "fromisphi": "FromIsPhi", "toisphi": "ToIsPhi"}

def normalise_columns(df):
    ren = {}
    for c in df.columns:
        k = str(c).strip().lower().replace("_", "").replace(" ", "")
        if k in _ALIAS: ren[c] = _ALIAS[k]
    df = df.rename(columns=ren)
    missing = [c for c in COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns {missing}. Found: {list(df.columns)}")
    return df[COLS]

def to_epoch_seconds(s):
    # Block timestamp -> integer unix seconds (handles unix s / ms / ns or datetime strings)
    if pd.api.types.is_numeric_dtype(s):
        v = s.astype("int64"); mx = int(v.max()) if len(v) else 0
        if mx > 10**17:   v = v // 10**9
        elif mx > 10**11: v = v // 10**3
        return v
    dt = pd.to_datetime(s, utc=True, errors="coerce")
    if dt.isna().any(): raise ValueError(f"{int(dt.isna().sum())} unparsable timestamps")
    return ((dt - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(seconds=1)).astype("int64")

print(pd.read_csv(DATA_PATH, nrows=5))

In [ ]:
CHUNK = 2_000_000

def scan_transactions(path):
    # Pass 1: keep all phishing-related rows; keep only (row_id, ts) of 0-0 rows.
    phi_parts, ids00, ts00, n_total = [], [], [], 0
    cats = {"0-0": 0, "0-1": 0, "1-0": 0, "1-1": 0}
    t0 = time.time()
    for chunk in pd.read_csv(path, chunksize=CHUNK):
        chunk = normalise_columns(chunk).copy()
        chunk["ts"] = to_epoch_seconds(chunk.pop("timestamp"))
        chunk["row_id"] = np.arange(n_total, n_total + len(chunk), dtype=np.int64)
        n_total += len(chunk)
        f = chunk["FromIsPhi"].astype("int8").to_numpy(); t = chunk["ToIsPhi"].astype("int8").to_numpy()
        for k, m in (("0-0", (f==0)&(t==0)), ("0-1", (f==0)&(t==1)), ("1-0", (f==1)&(t==0)), ("1-1", (f==1)&(t==1))):
            cats[k] += int(m.sum())
        is_phi = (f == 1) | (t == 1)
        phi_parts.append(chunk[is_phi])
        z = chunk[~is_phi]
        ids00.append(z["row_id"].to_numpy()); ts00.append(z["ts"].to_numpy())
        print(f"  scanned {n_total:,} rows ({time.time()-t0:.0f}s)", end="\r")
    print()
    df_phi = pd.concat(phi_parts, ignore_index=True)
    df_phi["FromIsPhi"] = df_phi["FromIsPhi"].astype("int8"); df_phi["ToIsPhi"] = df_phi["ToIsPhi"].astype("int8")
    pos00 = np.concatenate(ids00); day00 = np.concatenate(ts00) // DAY
    return df_phi, pos00, day00, cats, n_total

_scan_cache = Path(CACHE_DIR) / "scan_pass1.pkl"
if _scan_cache.exists():
    df_phi, pos00, day00, CATS, N_TOTAL = pickle.load(open(_scan_cache, "rb"))
    print("Loaded pass-1 cache.")
else:
    df_phi, pos00, day00, CATS, N_TOTAL = scan_transactions(DATA_PATH)
    pickle.dump((df_phi, pos00, day00, CATS, N_TOTAL), open(_scan_cache, "wb"))

print(f"Total transactions: {N_TOTAL:,}")

In [ ]:
# --- Table 1 -------------------------------------------------

t1 = pd.DataFrame({"this data": CATS})
display(t1)

d_min = pd.to_datetime(df_phi["ts"].min(), unit="s")
d_max = pd.to_datetime(df_phi["ts"].max(), unit="s")
print(f"Phishing-related transactions: {len(df_phi):,}")
print(f"Their time span: {d_min.date()} -> {d_max.date()}")
print("Missing values in phishing rows:", int(df_phi[["from", "to", "amount"]].isna().sum().sum()))

### 1.1 Sampling plans for the balanced datasets


In [ ]:
def d2day(s): return int(pd.Timestamp(s).timestamp() // DAY)

phi_day = (df_phi["ts"] // DAY).to_numpy()
def phishing_rows(start, end):
    m = (phi_day >= d2day(start)) & (phi_day <= d2day(end))
    return df_phi[m]

def candidates00(start, end):
    return pos00[(day00 >= d2day(start)) & (day00 <= d2day(end))]

PLANS = {}   # name -> dict(start, end, ids)
def add_plan(name, start, end, ids):
    PLANS[name] = dict(start=start, end=end, ids=np.sort(np.asarray(ids, dtype=np.int64)))

n_phi_window = len(phishing_rows(WINDOW_START, WINDOW_END))
cand_main = candidates00(WINDOW_START, WINDOW_END)
print(f"Phishing-related txns in window: {n_phi_window:,} | candidate 0-0 txns in window: {len(cand_main):,}")

# Datasets D0..D4: independent equal-size draws
for k in range(N_DATASETS):
    rng = np.random.default_rng(SEED + k)
    add_plan(f"D{k}", WINDOW_START, WINDOW_END, rng.choice(cand_main, min(n_phi_window, len(cand_main)), replace=False))

# Nested samples 
rng = np.random.default_rng(SEED + 1000)
_perm = rng.permutation(cand_main)[:max(GENUINE_TX_SIZES)]
for n in GENUINE_TX_SIZES:
    add_plan(f"S{n}", WINDOW_START, WINDOW_END, _perm[:n])

# T3 overlaps T1/T2 
TS_WINDOWS = {
    "TRAIN": ("2016-08-07", "2017-02-07"),
    "T1": ("2017-02-07", "2017-03-07"),
    "T2": ("2017-03-08", "2017-04-08"),
    "T3": ("2017-02-08", "2017-04-08"),
    "T4": ("2017-08-07", "2017-09-07"),
    "T5": ("2018-02-07", "2018-03-07"),
}
for i, (name, (a, b)) in enumerate(TS_WINDOWS.items()):
    n_p = len(phishing_rows(a, b)); cand = candidates00(a, b)
    rng = np.random.default_rng(SEED + 2000 + i)
    add_plan(name, a, b, rng.choice(cand, min(max(n_p, 1), len(cand)), replace=False) if len(cand) else [])
    print(f"  {name}: {a} -> {b} | phishing txns {n_p:,} | 0-0 candidates {len(cand):,}")

In [ ]:
# Pass 2: extract only the sampled 0-0 rows
def extract_rows(path, wanted):
    wanted = np.unique(wanted); parts, n_total = [], 0
    for chunk in pd.read_csv(path, chunksize=CHUNK):
        lo, hi = n_total, n_total + len(chunk); n_total = hi
        i0, i1 = np.searchsorted(wanted, [lo, hi])
        if i1 > i0:
            sub = normalise_columns(chunk.iloc[wanted[i0:i1] - lo]).copy()
            sub["ts"] = to_epoch_seconds(sub.pop("timestamp"))
            sub["row_id"] = wanted[i0:i1]
            parts.append(sub)
    out = pd.concat(parts, ignore_index=True)
    out["FromIsPhi"] = out["FromIsPhi"].astype("int8"); out["ToIsPhi"] = out["ToIsPhi"].astype("int8")
    return out.set_index("row_id")

_pool_cache = Path(CACHE_DIR) / "pool_pass2.pkl"
_wanted = np.unique(np.concatenate([p["ids"] for p in PLANS.values()]))
if _pool_cache.exists():
    POOL = pickle.load(open(_pool_cache, "rb"))
    if not np.isin(_wanted, POOL.index.to_numpy()).all():     # plans changed (e.g. QUICK_MODE toggled)
        POOL = extract_rows(DATA_PATH, _wanted); pickle.dump(POOL, open(_pool_cache, "wb"))
else:
    POOL = extract_rows(DATA_PATH, _wanted); pickle.dump(POOL, open(_pool_cache, "wb"))
print(f"Pool of sampled 0-0 rows: {len(POOL):,}")

def build_tx(plan):
    # Transactions of one balanced dataset: all phishing-related rows in the plan's window + its sampled 0-0 rows.
    p = PLANS[plan]
    tx = pd.concat([phishing_rows(p["start"], p["end"]).drop(columns="row_id"),
                    POOL.loc[p["ids"]]], ignore_index=True)
    tx["day"] = (tx["ts"] // DAY).astype("int64")
    tx["amount"] = tx["amount"].astype("float64")
    return tx


rows = []
for k in range(N_DATASETS):
    tx = build_tx(f"D{k}")
    addrs = pd.unique(pd.concat([tx["from"], tx["to"]], ignore_index=True))
    phi = set(tx.loc[tx.FromIsPhi == 1, "from"]) | set(tx.loc[tx.ToIsPhi == 1, "to"])
    rows.append(dict(dataset=f"D{k}", transactions=len(tx), addresses=len(addrs), phishing=len(phi), non_phishing=len(addrs) - len(phi)))
display(pd.DataFrame(rows))


## 2. Feature engineering

**Exactly 39 features, in three groups.** 



In [ ]:
# Feature engineering: (name, group, definition)
_F = [
 # ---------------- Transactional ----------------
 ("min_input_amt",   "Transactional", "Min I/P Amount",   "min amount received"),
 ("max_input_amt",   "Transactional", "Max I/P Amount",   "max amount received"),
 ("total_input_amt", "Transactional", "Total I/P Amount", "sum of amounts received"),
 ("avg_input_amt",   "Transactional", "Avg Input Amt",    "mean amount received"),
 ("min_output_amt",   "Transactional", "Min O/P Amount",   "min amount sent"),
 ("max_output_amt",   "Transactional", "Max O/P Amount",   "max amount sent"),
 ("total_output_amt", "Transactional", "Total O/P Amount", "sum of amounts sent"),
 ("avg_output_amt",   "Transactional", "Avg O/P Amount",   "mean amount sent"),
 ("diff_total_amounts", "Transactional", "Difference of Total Amounts", "total input - total output (balance)"),
 ("total_input_txns",  "Transactional", "Total No of Input Transactions",  "number of incoming transactions"),
 ("total_output_txns", "Transactional", "Total No of Output Transactions", "number of outgoing transactions"),
 ("max_input_txn_count_neighbor",  "Transactional", "Max Input occurrence / Highest I/P Txn Count", "highest number of incoming transactions from one neighbour"),
 ("min_input_txn_count_neighbor",  "Transactional", "Min Input occurrence / Lowest I/P Txn Count",  "lowest number of incoming transactions from one neighbour"),
 ("max_output_txn_count_neighbor", "Transactional", "Max O/P txn count / Highest O/P Txn Count",    "highest number of outgoing transactions to one neighbour"),
 ("min_output_txn_count_neighbor", "Transactional", "Min O/P txn count / Lowest O/P Txn Count",     "lowest number of outgoing transactions to one neighbour"),
 # ---------------- Structural  ----------------
 ("in_degree",  "Structural", "In degree",  "counts every incoming transaction"),
 ("out_degree", "Structural", "Out degree", "counts every outgoing transaction"),
 ("total_degree", "Structural", "Total Degree", "in + out"),
 ("avg_neighbor_degree", "Structural", "Avg_neigh_degree", "total degree of neighbours / number of neighbours"),
 ("core_number", "Structural", "Core Number", "nx.core_number (undirected simple graph)"),
 ("clustering_coeff", "Structural", "Clus_Coeff", "nx.clustering (undirected simple graph)"),
 ("eigenvector_centrality", "Structural", "Eig_cen", "nx.eigenvector_centrality (directed)"),
 ("closeness_centrality",   "Structural", "Clo_cen", "nx.closeness_centrality (directed)"),
 ("harmonic_centrality",    "Structural", "Harmo_cen", "nx.harmonic_centrality (directed)"),
 ("pagerank",               "Structural", "Page Rank", "nx.pagerank (directed)"),
 ("local_reaching_centrality", "Structural", "loc_rech_cen", "nx.local_reaching_centrality (directed)"),
 ("hits",                   "Structural", "Hits", "nx.hits (HITS_MODE: authority | hub)"),
 # ---------------- Temporal----------------
 ("input_lifetime",  "Temporal", "Input Lifetime",  "days between first and last incoming transaction"),
 ("output_lifetime", "Temporal", "Output Lifetime", "days between first and last outgoing transaction"),
 ("distinct_input_timestamps",  "Temporal", "No of distinct input timestamp",  "DT_i for incoming transactions"),
 ("distinct_output_timestamps", "Temporal", "No of distinct output timestamp", "DT_i for outgoing transactions"),
 ("max_input_txn_count_ts",  "Temporal", "Highest freq of input timestamp",  "max_t C_i^t, incoming transactions per timestamp"),
 ("min_input_txn_count_ts",  "Temporal", "Lowest freq of input timestamp",   "min_t C_i^t, incoming transactions per timestamp"),
 ("max_output_txn_count_ts", "Temporal", "Highest freq of output timestamp", "max_t C_i^t, outgoing transactions per timestamp"),
 ("min_output_txn_count_ts", "Temporal", "Lowest freq of output timestamp",  "min_t C_i^t, outgoing transactions per timestamp"),
 ("input_ts_txn_activity",  "Temporal", "I/P TS-TXN Activity", "input side"),
 ("output_ts_txn_activity", "Temporal", "O/P TS-TXN Activity", "output side"),
 ("input_ts_txn_ratio",  "Temporal", "Ratio of input timestamp",  "input side"),
 ("output_ts_txn_ratio", "Temporal", "Ratio of output timestamp", "output side"),
]
FEATURE_INFO = pd.DataFrame(_F, columns=["feature", "group", "article label (Fig. 8 / 12 / Table 5)", "definition used here"])
TRANSACTIONAL = FEATURE_INFO.query("group=='Transactional'").feature.tolist()
STRUCTURAL    = FEATURE_INFO.query("group=='Structural'").feature.tolist()
TEMPORAL      = FEATURE_INFO.query("group=='Temporal'").feature.tolist()
FEATURES = TRANSACTIONAL + STRUCTURAL + TEMPORAL
assert len(FEATURES) == len(set(FEATURES)) == 39, len(FEATURES)
print({"Transactional": len(TRANSACTIONAL), "Structural": len(STRUCTURAL), "Temporal": len(TEMPORAL), "Total": len(FEATURES)})
display(FEATURE_INFO)

In [ ]:
from scipy.sparse.csgraph import dijkstra

# --------------------------Transactional--------------------
def transactional_features(tx, nodes):
    inc = tx.groupby("to")["amount"].agg(min_input_amt="min", max_input_amt="max", total_input_amt="sum",
                                          avg_input_amt="mean", total_input_txns="count")
    out = tx.groupby("from")["amount"].agg(min_output_amt="min", max_output_amt="max", total_output_amt="sum",
                                            avg_output_amt="mean", total_output_txns="count")
    nb_in  = tx.groupby(["to", "from"]).size().groupby(level=0).agg(max_input_txn_count_neighbor="max", min_input_txn_count_neighbor="min")
    nb_out = tx.groupby(["from", "to"]).size().groupby(level=0).agg(max_output_txn_count_neighbor="max", min_output_txn_count_neighbor="min")
    F = pd.DataFrame(index=nodes).join([inc, out, nb_in, nb_out]).fillna(0.0)
    F["diff_total_amounts"] = F["total_input_amt"] - F["total_output_amt"]
    return F[TRANSACTIONAL]

# -------------------------- Structural--------------------
def path_centralities(G, batch=256):
    """closeness (nx default, wf_improved), harmonic and local reaching centrality in ONE batched pass (C code).
    Returns the same values as nx.closeness_centrality(G), nx.harmonic_centrality(G) and
    nx.local_reaching_centrality(G, v) (unweighted directed) -- verified against NetworkX on random graphs."""
    nl = list(G); N = len(nl)
    A = nx.to_scipy_sparse_array(G, nodelist=nl, weight=None, dtype="float64").tocsr()
    cnt_in = np.zeros(N); sum_in = np.zeros(N); harm = np.zeros(N); reach_out = np.zeros(N)
    for s in range(0, N, batch):
        idx = np.arange(s, min(s + batch, N))
        D = dijkstra(A, directed=True, unweighted=True, indices=idx)   # D[i, j] = dist(idx[i] -> j)
        pos = np.isfinite(D) & (D > 0)                                  # reachable, excluding the source itself
        reach_out[idx] = pos.sum(1)                                     # nodes reachable FROM each source
        cnt_in += pos.sum(0)                                            # sources that can reach each node
        sum_in += np.where(pos, D, 0.0).sum(0)
        with np.errstate(divide="ignore"):
            harm += np.where(pos, 1.0 / D, 0.0).sum(0)
    den = max(N - 1, 1)
    clo = np.where(sum_in > 0, (cnt_in / np.where(sum_in > 0, sum_in, 1)) * (cnt_in / den), 0.0)
    return dict(zip(nl, clo)), dict(zip(nl, harm)), dict(zip(nl, reach_out / den))

def _hits_robust(G):
    # nx.hits can raise PowerIterationFailedConvergence on small / degenerate graphs. Retry, then fall back to a
    # plain power iteration on the same HITS update (a <- A^T h, h <- A a, L1-normalised) and keep the final iterate.
    for max_iter, tol in ((1000, 1e-8), (10000, 1e-6)):
        try:
            return nx.hits(G, max_iter=max_iter, tol=tol)
        except nx.PowerIterationFailedConvergence:
            pass
    print("\n    [warn] nx.hits did not converge; using a fixed-iteration power-iteration fallback for this graph")
    nl = list(G); A = nx.to_scipy_sparse_array(G, nodelist=nl, weight=None, dtype="float64")
    h = np.full(len(nl), 1.0 / max(len(nl), 1))
    for _ in range(2000):
        a = A.T @ h; a /= max(a.sum(), 1e-300)
        h_new = A @ a; h_new /= max(h_new.sum(), 1e-300)
        if np.abs(h_new - h).sum() < 1e-10: h = h_new; break
        h = h_new
    a = A.T @ h; a /= max(a.sum(), 1e-300)
    return dict(zip(nl, h)), dict(zip(nl, a))

def _eigenvector_robust(G, U):
    try:
        return nx.eigenvector_centrality(G, max_iter=1000, tol=1e-6)
    except nx.PowerIterationFailedConvergence:
        print("\n    [warn] eigenvector centrality did not converge on the directed graph; using the undirected graph for this feature")
    try:
        return nx.eigenvector_centrality(U, max_iter=5000, tol=1e-6)
    except nx.PowerIterationFailedConvergence:
        return nx.eigenvector_centrality_numpy(U)

def structural_features(tx, nodes):
    t0 = time.time()
    F = pd.DataFrame(index=nodes)
    # Degrees: count every transaction (multi-edges)
    F["in_degree"]    = tx.groupby("to").size().reindex(nodes).fillna(0).to_numpy()
    F["out_degree"]   = tx.groupby("from").size().reindex(nodes).fillna(0).to_numpy()
    F["total_degree"] = F["in_degree"] + F["out_degree"]

    # Simple directed graph (duplicates collapsed, self-loops dropped)
    e = tx.loc[tx["from"] != tx["to"], ["from", "to"]].drop_duplicates()
    G = nx.DiGraph(); G.add_nodes_from(nodes); G.add_edges_from(e.itertuples(index=False, name=None))
    U = G.to_undirected()

    # average neighbour degree = total degree of the neighbours / number of neighbours
    und = pd.concat([e.rename(columns={"from": "u", "to": "v"}), e.rename(columns={"from": "v", "to": "u"})]).drop_duplicates()
    und["dv"] = F["total_degree"].reindex(und["v"]).to_numpy()
    s = und.groupby("u")["dv"].agg(["sum", "count"])
    F["avg_neighbor_degree"] = (s["sum"] / s["count"]).reindex(nodes).fillna(0).to_numpy()

    F["core_number"]      = pd.Series(nx.core_number(U)).reindex(nodes).to_numpy()
    F["clustering_coeff"] = pd.Series(nx.clustering(U)).reindex(nodes).to_numpy()
    print(f"    degrees/core/clustering: {time.time()-t0:.0f}s", end=" | ")

    F["eigenvector_centrality"] = pd.Series(_eigenvector_robust(G, U)).reindex(nodes).to_numpy()
    F["pagerank"] = pd.Series(nx.pagerank(G, max_iter=1000)).reindex(nodes).to_numpy()
    h, a = _hits_robust(G)
    F["hits"] = pd.Series(a if HITS_MODE == "authority" else h).reindex(nodes).to_numpy()
    print(f"eig/pagerank/hits: {time.time()-t0:.0f}s", end=" | ")

    # closeness, harmonic, local reaching: one fast batched pass (same values as the NetworkX functions)
    clo, har, loc = path_centralities(G)
    F["closeness_centrality"]      = pd.Series(clo).reindex(nodes).to_numpy()
    F["harmonic_centrality"]       = pd.Series(har).reindex(nodes).to_numpy()
    F["local_reaching_centrality"] = pd.Series(loc).reindex(nodes).to_numpy()
    print(f"path-based centralities: {time.time()-t0:.0f}s")
    return F[STRUCTURAL]

# --------------------Temporal--------------------
def temporal_features(tx, nodes):
    tx = tx.assign(ts_key=tx["ts"] if TS_GRANULARITY == "second" else tx["day"])
    F = pd.DataFrame(index=nodes)
    for side, col in (("input", "to"), ("output", "from")):
        g = tx.groupby(col)
        n    = g.size()                                    # sum_t C_i^t
        life = g["day"].max() - g["day"].min()             # lifetime in days
        dt   = g["ts_key"].nunique()                       # DT_i
        per_ts = tx.groupby([col, "ts_key"]).size().groupby(level=0).agg(["max", "min"])   # C_i^t statistics
        F[f"{side}_lifetime"]             = life
        F[f"distinct_{side}_timestamps"]  = dt
        F[f"max_{side}_txn_count_ts"]     = per_ts["max"]
        F[f"min_{side}_txn_count_ts"]     = per_ts["min"]
        F[f"{side}_ts_txn_activity"] = dt * n if ACTIVITY_FORMULA == "as_printed" else n / dt          # Eq. 6
        F[f"{side}_ts_txn_ratio"]    = dt / n if RATIO_FORMULA == "eq7" else n / dt                    # Eq. 7
    return F.reindex(nodes).fillna(0.0)[TEMPORAL]

# ---------------------------------------------------------------- 2.4 Assemble
def build_feature_table(tx, verbose=True):
    nodes = pd.Index(pd.unique(pd.concat([tx["from"], tx["to"]], ignore_index=True)))
    phishing = set(tx.loc[tx.FromIsPhi == 1, "from"]) | set(tx.loc[tx.ToIsPhi == 1, "to"])
    if verbose: print(f"  {len(nodes):,} addresses, {len(tx):,} transactions")
    T = transactional_features(tx, nodes)
    S = structural_features(tx, nodes)
    P = temporal_features(tx, nodes)
    X = pd.concat([T, S, P], axis=1)[FEATURES].astype("float64")
    y = pd.Series(nodes.isin(list(phishing)).astype("int8"), index=nodes, name="label")
    assert list(X.columns) == FEATURES
    return X, y

def get_features(plan):
    f = Path(CACHE_DIR) / f"features_{plan}_{HITS_MODE}_{TS_GRANULARITY}_{ACTIVITY_FORMULA}_{RATIO_FORMULA}.pkl"
    if f.exists():
        return pd.read_pickle(f)
    print(f"[{plan}] computing features")
    X, y = build_feature_table(build_tx(plan))
    pd.to_pickle((X, y), f)
    return X, y

### 2.5 Compute features for D₀ … 

The structural block (closeness, harmonic, local-reaching centrality) is the slow part and its cost grows faster than linearly with graph size. In my synthetic test it took ~30 s at 12 k transactions and ~4 min at 37 k, so on the full ~120 k-transaction datasets expect it to be **much slower (possibly an hour or more per dataset)**. Put `CACHE_DIR` on Drive so this is paid once. In `QUICK_MODE` only D₀ is computed.

In [ ]:
FEATS = {}
for k in range(N_DATASETS):
    FEATS[k] = get_features(f"D{k}")

X0, y0 = FEATS[0]
print("D0 feature matrix:", X0.shape, "| phishing:", int(y0.sum()), "| non-phishing:", int((y0 == 0).sum()),
      f"| phishing share: {y0.mean():.2%}")
assert np.isfinite(X0.to_numpy()).all(), "non-finite values in features"
assert X0.shape[1] == 39
display(X0.groupby(y0.values).mean().T.rename(columns={0: "non-phishing mean", 1: "phishing mean"}))

In [ ]:
import os; print(os.listdir(CACHE_DIR))

## 3. Classification experiments 



*Protocol* : random 80:20 split, and 10-fold CV repeated 10 times. SMOTE / random under-sampling are applied **to the training portion only**. Reported-results mapping: Tables 6 and 9 are unsampled (80:20 and CV); Tables 10 and 11 are SMOTE / random under-sampling on the 80:20 split only. Classifier hyper-parameters was set to defaults.

In [ ]:
MODEL_ORDER = ["Random Forest", "XGBoost", "Ada Boost", "Cat Boost", "Light GBM", "SVM", "MLP", "Logistic Regression"]
def make_model(name, seed=SEED):
    m = {
        "Random Forest":       lambda: RandomForestClassifier(random_state=seed, n_jobs=-1),
        "XGBoost":             lambda: XGBClassifier(random_state=seed, eval_metric="logloss", n_jobs=-1),
        "Ada Boost":           lambda: AdaBoostClassifier(random_state=seed),
        "Cat Boost":           lambda: CatBoostClassifier(random_state=seed, verbose=0, thread_count=-1),
        "Light GBM":           lambda: LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1),
        "SVM":                 lambda: SVC(random_state=seed),
        "MLP":                 lambda: MLPClassifier(random_state=seed),
        "Logistic Regression": lambda: LogisticRegression(max_iter=1000, random_state=seed),
    }[name]()
    if SCALE_LINEAR_MODELS and name in ("SVM", "MLP", "Logistic Regression"):
        m = make_pipeline(StandardScaler(), m)
    return m

def model_score(model, X):
    return model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X)

def metrics(y, pred, score):
    return dict(precision=precision_score(y, pred, zero_division=0), recall=recall_score(y, pred, zero_division=0),
                f1=f1_score(y, pred, zero_division=0),
                auc=roc_auc_score(y, score) if len(np.unique(y)) > 1 else np.nan,
                macro_f1=f1_score(y, pred, average="macro", zero_division=0))

def resample(X, y, kind, seed):
    if kind is None:   return X, y
    if kind == "smote": return SMOTE(random_state=seed).fit_resample(X, y)
    if kind == "rus":   return RandomUnderSampler(random_state=seed).fit_resample(X, y)
    raise ValueError(kind)

def fit_eval(name, Xtr, ytr, Xte, yte, sampling=None, seed=SEED):
    Xr, yr = resample(Xtr, ytr, sampling, seed)
    model = make_model(name, seed).fit(Xr, yr)
    return metrics(yte, model.predict(Xte), model_score(model, Xte))

def split(X, y, seed):
    return train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)

def run_split_experiment(sampling=None, models=MODEL_ORDER):
    rows = []
    for k, (X, y) in FEATS.items():
        Xtr, Xte, ytr, yte = split(X, y, SEED + k)
        for name in models:
            t0 = time.time()
            rows.append(dict(dataset=f"D{k}", model=name, **fit_eval(name, Xtr, ytr, Xte, yte, sampling, SEED + k)))
            print(f"    D{k} {name:<20} F1={rows[-1]['f1']:.3f}  ({time.time()-t0:.0f}s)")
    df = pd.DataFrame(rows)
    return df, df.groupby("model")[["precision", "recall", "f1", "auc", "macro_f1"]].mean().reindex(models)

RESULTS = {}

### 80:20 split, no sampling

In [ ]:
raw6, RESULTS["table6"] = run_split_experiment(sampling=None)
display(RESULTS["table6"].round(2))

### 80:20 split with SMOTE over-sampling / random under-sampling (training set only)

In [ ]:
raw10, RESULTS["table10_smote"] = run_split_experiment(sampling="smote")
print("SMOTE over-sampling (Table 10)"); display(RESULTS["table10_smote"].round(2))
raw11, RESULTS["table11_rus"] = run_split_experiment(sampling="rus")
print("Random under-sampling (Table 11)"); display(RESULTS["table11_rus"].round(2))

### 10-fold cross-validation (repeated)

The 10-fold CV 10 times; `CV_REPEATS` controls this. In `QUICK_MODE` the (slow, unscaled) SVM is skipped here.

In [ ]:
def run_cv(sampling=None, models=MODEL_ORDER):
    rows = []
    for k, (X, y) in FEATS.items():
        rskf = RepeatedStratifiedKFold(n_splits=CV_FOLDS, n_repeats=CV_REPEATS, random_state=SEED + k)
        for name in models:
            t0 = time.time(); fold = []
            for tr, te in rskf.split(X, y):
                fold.append(fit_eval(name, X.iloc[tr], y.iloc[tr], X.iloc[te], y.iloc[te], sampling, SEED + k))
            rows.append(dict(dataset=f"D{k}", model=name, **pd.DataFrame(fold).mean().to_dict()))
            print(f"    D{k} {name:<20} F1={rows[-1]['f1']:.3f}  ({time.time()-t0:.0f}s)")
    df = pd.DataFrame(rows)
    return df.groupby("model")[["precision", "recall", "f1", "auc", "macro_f1"]].mean().reindex([m for m in models if m in set(df.model)])

cv_models = [m for m in MODEL_ORDER if not (QUICK_MODE and m == "SVM")]
RESULTS["table9_cv"] = run_cv(None, cv_models)
display(RESULTS["table9_cv"].round(2))

### 3.4 Comparison with the article (Table 6, XGBoost row and overall pattern)

Article Table 6 (80:20, no sampling): XGBoost 0.98 / 0.98 / 0.98 / 0.98; SVM 0.85 / 0.59 / 0.64 / 0.59; MLP 0.59 / 0.69 / 0.62 / 0.69; Logistic Regression 0.48 / 0.37 / 0.41 / 0.37. The article's claim is that tree ensembles cope with the class imbalance while SVM / MLP / Logistic Regression do not, and that sampling (especially SMOTE) removes the gap.

In [ ]:
PAPER_T6 = pd.DataFrame({
    "Random Forest": (0.97, 0.96, 0.97, 0.96), "XGBoost": (0.98, 0.98, 0.98, 0.98), "Ada Boost": (0.95, 0.97, 0.96, 0.97),
    "Cat Boost": (0.97, 0.98, 0.97, 0.97), "Light GBM": (0.97, 0.98, 0.97, 0.97), "SVM": (0.85, 0.59, 0.64, 0.59),
    "MLP": (0.59, 0.69, 0.62, 0.69), "Logistic Regression": (0.48, 0.37, 0.41, 0.37)},
    index=["precision", "recall", "f1", "auc"]).T
cmp6 = RESULTS["table6"][["precision", "recall", "f1", "auc"]].join(PAPER_T6, rsuffix="_paper").round(2)
display(cmp6)

## 4. Run-to-run variation and statistical tests

Table below, for each of 10 trials, the mean/variance/SD of recall (in %) over 10 repeated runs of the catboost model on different random train-test splits. Fig. below plots the SDs.

# statistical tests
paired *t*-test and Mann–Whitney *U* test compare the proposed features against **SIGTRAN** (a baseline). If you have 10 F1 values for a baseline, put them in `BASELINE_F1_RUNS` and the tests run on them; otherwise that cell is skipped.

In [ ]:
X, y = FEATS[0]
rows, f1_runs = [], []
for trial in range(TRIAL_COUNT):
    rec, f1s = [], []
    for run in range(RUNS_PER_TRIAL):
        s = SEED + 1000 * trial + run
        Xtr, Xte, ytr, yte = split(X, y, s)
        m = fit_eval("Cat Boost", Xtr, ytr, Xte, yte, None, s)
        rec.append(100 * m["recall"]); f1s.append(100 * m["f1"])
    rows.append(dict(trial=trial + 1, mean=np.mean(rec), variance=np.var(rec), sd=np.std(rec)))
    f1_runs.append(np.mean(f1s))
TABLE8 = pd.DataFrame(rows)
display(TABLE8.round(4))
ax = TABLE8.plot.bar(x="trial", y="sd", legend=False, color="lightgreen", edgecolor="k", figsize=(7, 3.5))
ax.set_ylabel("Standard deviation of recall (%)"); ax.set_xlabel("Trial"); ax.set_title("Recall variation across experiments (cf. Fig. 7)")
plt.tight_layout(); plt.show()

In [ ]:
# Optional: statistical tests against a baseline's 10 F1 runs.  Values in %, same unit as below.
BASELINE_F1_RUNS = None            # e.g. [93.1, 92.8, ...]  (needs the same number of runs as `ours`)
ours = np.array(f1_runs)
if BASELINE_F1_RUNS is None:
    print("Statistical tests skipped: set BASELINE_F1_RUNS to compare against a baseline (article: SIGTRAN).")
    print(f"Proposed features, F1 over {len(ours)} runs: mean {ours.mean():.2f}%, variance {ours.var():.3f}, median {np.median(ours):.2f}%")
else:
    base = np.array(BASELINE_F1_RUNS); assert len(base) == len(ours), "need equal run counts for the paired test"
    t = stats.ttest_rel(ours, base, alternative="greater")
    u = stats.mannwhitneyu(ours, base, alternative="greater")
    print(f"Paired t-test (H1: proposed > baseline):  t={t.statistic:.3f}, p={t.pvalue:.3g}")
    print(f"Mann-Whitney U (H1: proposed > baseline): U={u.statistic:.1f}, p={u.pvalue:.3g}")

## 5. Ablation and feature analysis

### 5.1 Individual feature ranking 
Random Forest trained on **one feature at a time**, no sampling, 80:20 split; features ranked by class-1 (phishing) recall.

In [ ]:
X, y = FEATS[0]
Xtr, Xte, ytr, yte = split(X, y, SEED)
rank = {}
for f in FEATURES:
    rf = RandomForestClassifier(random_state=SEED, n_jobs=-1).fit(Xtr[[f]], ytr)
    rank[f] = recall_score(yte, rf.predict(Xte[[f]]), zero_division=0)
RANK = pd.Series(rank).sort_values(ascending=False)
TOP10 = RANK.index[:10].tolist()

fig, ax = plt.subplots(figsize=(13, 4))
RANK.plot.bar(ax=ax, color="lightgreen", edgecolor="k")
ax.set_ylabel("Individual performance (recall, class 1)"); ax.set_title("Ablation: single-feature Random Forest (cf. Fig. 8)")
plt.tight_layout(); plt.show()

print("Top-10 here:", TOP10)
display(RANK.head(10).to_frame("recall").join(FEATURE_INFO.set_index("feature")["group"]))

### 5.2 Phishing vs non-phishing averages of the top-10 features.

In [ ]:
means = X[TOP10].groupby(y.values).mean().T.rename(columns={0: "Non-Phishing", 1: "Phishing"})
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, f in zip(axes.ravel(), TOP10):
    ax.bar(["Phishing", "Non-Phishing"], [means.loc[f, "Phishing"], means.loc[f, "Non-Phishing"]], color=["#ff4d4d", "#4da64d"], edgecolor="k")
    ax.set_title(f, fontsize=9); ax.tick_params(axis="x", labelsize=8)
fig.suptitle("Average value of the top-10 features by class (cf. Figs. 9-10)"); plt.tight_layout(); plt.show()

### 5.3 Feature-group contribution 
catboost on each group and every combination of groups (80:20, no sampling).

In [ ]:
G = {"Trans": TRANSACTIONAL, "Struct": STRUCTURAL, "Temp": TEMPORAL}
combos = {"Trans": ["Trans"], "Struct": ["Struct"], "Temp": ["Temp"], "Trans+Struct": ["Trans", "Struct"],
          "Struct+Temp": ["Struct", "Temp"], "Trans+Temp": ["Trans", "Temp"], "Trans+Struct+Temp": ["Trans", "Struct", "Temp"]}
rows = []
for label, parts in combos.items():
    cols = [f for p in parts for f in G[p]]
    m = fit_eval("Cat Boost", Xtr[cols], ytr, Xte[cols], yte, None, SEED)
    rows.append(dict(feature_set=label, n_features=len(cols), recall=m["recall"], precision=m["precision"], f1=m["f1"]))
GROUPS = pd.DataFrame(rows).set_index("feature_set")
display(GROUPS.round(3))
GROUPS[["recall", "precision", "f1"]].mul(100).plot.bar(figsize=(10, 4), edgecolor="k", ylabel="Performance (%)", title="Feature-set contributions (cf. Fig. 11)")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

### 5.4 Correlation study 

In [ ]:
corr = X.corr()
plt.figure(figsize=(16, 14))
sns.heatmap(corr, cmap="rocket", vmin=-1, vmax=1, square=True,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            xticklabels=True, yticklabels=True, cbar_kws={"shrink": .7})
plt.title("Feature correlation (cf. Fig. 12)"); plt.tight_layout(); plt.show()

iu = np.triu_indices_from(corr, k=1)
n_high = int((corr.to_numpy()[iu] > 0.7).sum())
print(f"Pairs with correlation > 0.7 (the article's 'high correlation' threshold): {n_high} of {len(iu[0])}")
top10_corr = corr.loc[TOP10, TOP10].abs().to_numpy()[np.triu_indices(10, 1)]
print(f"Max |corr| among the top-10 features: {top10_corr.max():.2f}")

## 6. Temporal stability and effect of genuine transactions 

### 6.1 Temporal stability 
Train on **2016/08/07 – 2017/02/07** and test on five later windows T₁…T₅. For each window, features are recomputed from that window's transactions: all phishing-related transactions plus an equal number of randomly drawn 0–0 transactions.

In [ ]:
rows = []
Xa, ya = get_features("TRAIN")
print(f"Training window: {len(Xa):,} addresses, {int(ya.sum())} phishing")
clf = make_model("XGBoost", SEED).fit(Xa, ya)
for name in ["T1", "T2", "T3", "T4", "T5"]:
    Xt, yt = get_features(name)
    if yt.sum() == 0:
        print(f"{name}: no phishing addresses in this window, skipped"); continue
    m = metrics(yt, clf.predict(Xt), model_score(clf, Xt))
    rows.append(dict(period=name, window=" - ".join(TS_WINDOWS[name]), phishing=int(yt.sum()), **m))
TEMPORAL_STABILITY = pd.DataFrame(rows).set_index("period")
display(TEMPORAL_STABILITY.round(3))
TEMPORAL_STABILITY[["precision", "recall", "f1"]].mul(100).plot.bar(figsize=(8, 4), edgecolor="k", ylabel="Performance (%)",
                                                                   ylim=(0, 105), title="Temporal stability (cf. Fig. 13)")
plt.tight_layout(); plt.show()

### 6.2 Effect of the number of genuine (0–0) transactions
The phishing transactions are held fixed and the number of 0–0 transactions is varied over 5 000 … 100 000. Each setting: features recomputed, 80:20 split, catboost.

In [ ]:
rows = []
for n in GENUINE_TX_SIZES:
    Xn, yn = get_features(f"S{n}")
    Xtr_, Xte_, ytr_, yte_ = split(Xn, yn, SEED)
    m = fit_eval("Cat Boost", Xtr_, ytr_, Xte_, yte_, None, SEED)
    rows.append(dict(genuine_txns=n, addresses=len(Xn), phishing=int(yn.sum()), **m))
GENUINE = pd.DataFrame(rows).set_index("genuine_txns")
display(GENUINE.round(3))
GENUINE[["recall", "precision", "f1"]].mul(100).plot.bar(figsize=(10, 4), edgecolor="k", ylim=(0, 105), ylabel="Performance (%)",
                                                       title="Effect of the number of genuine transactions (cf. Fig. 14)")
plt.tight_layout(); plt.show()

## 7. GCN and GIN with and without the proposed features

Evaluates GCN and GIN on this task with and without its engineered features and with 2–8 layers, and ultimately prefers catboost. 

| Item | Choice made here |
|---|---|
| Graph | The sampled D₀ transactions as a graph over the same addresses as the feature table; duplicate edges collapsed; `GNN_SYMMETRISE=True` makes message passing use both directions (PyG's `GCNConv` on a directed `edge_index` only aggregates from in-neighbours) |
| Input **without** proposed features | A constant 1-dimensional feature (the model sees topology only) |
| Input **with** proposed features | The 39 engineered features, standardised using **training nodes only** (`GNN_LOG1P` optionally applies log1p first) |
| Classifier | The GNN itself (linear head on the last layer). The article also tried GNN + XGBoost and found XGBoost better; that variant is not implemented here |
| Split / imbalance | Stratified 80:20 node split; class-weighted cross-entropy (SMOTE has no graph analogue) |
| Training | Adam, lr 0.01, weight decay 5e-4, hidden 64, dropout 0.5, fixed number of epochs, no early stopping; test metrics from the final epoch (no test-set selection) |
| Layers | GCNConv / GINConv (MLP of 2 linear layers, trainable ε) × L, L = 2…8 |



In [ ]:
try:
    import torch, torch.nn as nn, torch.nn.functional as Fnn
    from torch_geometric.nn import GCNConv, GINConv
    from torch_geometric.utils import to_undirected
    HAS_PYG = True
except Exception as e:
    HAS_PYG = False
    print("PyTorch Geometric unavailable -> Part 7 skipped:", e)

if HAS_PYG:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", DEVICE)

    X, y = FEATS[0]
    tx0 = build_tx("D0")
    idx = pd.Series(np.arange(len(X)), index=X.index)
    e = tx0.loc[tx0["from"] != tx0["to"], ["from", "to"]].drop_duplicates()
    edge_index = torch.tensor(np.vstack([idx.reindex(e["from"]).to_numpy(), idx.reindex(e["to"]).to_numpy()]), dtype=torch.long)
    if GNN_SYMMETRISE: edge_index = to_undirected(edge_index)
    edge_index = edge_index.to(DEVICE)
    print(f"Graph: {len(X):,} nodes, {edge_index.shape[1]:,} (directed) edge entries, {int(y.sum())} phishing nodes")

    tr_idx, te_idx = train_test_split(np.arange(len(X)), test_size=TEST_SIZE, stratify=y.to_numpy(), random_state=SEED)
    Xf = X.to_numpy(dtype="float64")
    if GNN_LOG1P: Xf = np.log1p(np.clip(Xf, 0, None))
    sc = StandardScaler().fit(Xf[tr_idx])                       # fit on training nodes only -> no leakage
    X_with    = torch.tensor(sc.transform(Xf), dtype=torch.float32).to(DEVICE)
    X_without = torch.ones((len(X), 1), dtype=torch.float32, device=DEVICE)
    y_t = torch.tensor(y.to_numpy(), dtype=torch.long, device=DEVICE)
    tr_t, te_t = torch.tensor(tr_idx, device=DEVICE), torch.tensor(te_idx, device=DEVICE)

    n_pos = int(y.iloc[tr_idx].sum()); n_neg = len(tr_idx) - n_pos
    CLASS_W = torch.tensor([1.0, n_neg / max(n_pos, 1)], dtype=torch.float32, device=DEVICE)

    class GNN(nn.Module):
        def __init__(self, kind, in_dim, hidden, layers, dropout):
            super().__init__()
            self.convs = nn.ModuleList()
            for i in range(layers):
                d = in_dim if i == 0 else hidden
                if kind == "GCN":
                    self.convs.append(GCNConv(d, hidden))
                else:
                    self.convs.append(GINConv(nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, hidden)), train_eps=True))
            self.head, self.dropout = nn.Linear(hidden, 2), dropout
        def forward(self, x, ei):
            for conv in self.convs:
                x = Fnn.dropout(Fnn.relu(conv(x, ei)), self.dropout, self.training)
            return self.head(x)

    def train_gnn(kind, x, layers, seed):
        torch.manual_seed(seed); np.random.seed(seed)
        model = GNN(kind, x.shape[1], GNN_HIDDEN, layers, GNN_DROPOUT).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=GNN_LR, weight_decay=GNN_WD)
        for _ in range(GNN_EPOCHS):
            model.train(); opt.zero_grad()
            loss = Fnn.cross_entropy(model(x, edge_index)[tr_t], y_t[tr_t], weight=CLASS_W)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            logits = model(x, edge_index)[te_t]
        prob = torch.softmax(logits, 1)[:, 1].cpu().numpy()
        return metrics(y.iloc[te_idx].to_numpy(), logits.argmax(1).cpu().numpy(), prob)

### Layer sweep, with and without the proposed features.

In [ ]:
if HAS_PYG:
    rows = []
    for kind in ["GCN", "GIN"]:
        for fname, x in (("without features", X_without), ("with proposed features", X_with)):
            for L in GNN_LAYERS:
                t0 = time.time()
                ms = [train_gnn(kind, x, L, SEED + s) for s in range(GNN_SEEDS)]
                rows.append(dict(model=kind, inputs=fname, layers=L, **pd.DataFrame(ms).mean().to_dict()))
                print(f"  {kind:<3} | {fname:<22} | L={L} | F1={rows[-1]['f1']:.3f} ({time.time()-t0:.0f}s)")
    GNN_RES = pd.DataFrame(rows)
    display(GNN_RES.round(3))

In [ ]:
if HAS_PYG:
    # Fig. 16 analogue: F1 vs number of layers  (dashed = article, approximate readings from the figure)
    PAPER_FIG16 = {("GCN", "without features"): [49, 49, 50, 50, 48, 46, 46], ("GCN", "with proposed features"): [82, 83, 83, 83, 83, 83, 83],
                   ("GIN", "without features"): [58, 60, 60, 58, 56, 56, 55]}
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for (kind, inp), d in GNN_RES.groupby(["model", "inputs"]):
        ln, = ax.plot(d["layers"], 100 * d["f1"], marker="o", label=f"{kind} - {inp}")
        if (kind, inp) in PAPER_FIG16 and len(d) == 7:
            ax.plot(d["layers"], PAPER_FIG16[(kind, inp)], ls="--", color=ln.get_color(), alpha=.5)
    ax.set_xlabel("Number of layers"); ax.set_ylabel("F1-score (%)"); ax.set_title("GNN depth analysis (cf. Fig. 16; dashed = article, approx.)")
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()

    # Fig. 15 analogue at 2 layers + XGBoost reference
    two = GNN_RES[GNN_RES.layers == min(GNN_LAYERS)].set_index(["model", "inputs"])[["precision", "recall", "f1"]] * 100
    ref = (RESULTS["table6"].loc["Cat Boost", ["precision", "recall", "f1"]] * 100).to_frame().T
    ref.index = pd.MultiIndex.from_tuples([("Cat Boosts", "with proposed features")], names=two.index.names)
    two = pd.concat([two, ref])
    display(two.round(1))
    two.plot.bar(figsize=(10, 4), edgecolor="k", ylabel="Performance (%)", title=f"GNN performance, {min(GNN_LAYERS)} layers (cf. Fig. 15)")
    plt.xticks(rotation=25, ha="right"); plt.tight_layout(); plt.show()

## 8. Summary and export